In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
import random, re
from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM

from msnap.utils import common_utils, json_utils, container_utils, file_utils, model_utils, tokenizer_utils
from msnap.utils.container_utils import chunks
from msnap.core import msnap_prompts

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/msnap'
data_dir = f'{work_dir}/data'

# zero_shot_target = 'fact'
zero_shot_target = 'other'

in_dir = f'{data_dir}/check_contexts/zero_shot_{zero_shot_target}'
out_dir = f'{data_dir}/check_targets/zero_shot_{zero_shot_target}'

In [ ]:
model_name = 'Llama-3.2-3B'
# model_name = 'Llama-3.1-8B'

model_name_or_path = f'meta-llama/{model_name}-Instruct'
dtype = 'bfloat16'
device = 'cuda:0'
max_seq_length = 4096
max_new_tokens = 64

model: AutoModelForCausalLM = model_utils.get_model(model_name_or_path, dtype, device, is_eval=True)

# 평가/추론 시에는 반드시 'left' 패딩
tokenizer: PreTrainedTokenizerFast = tokenizer_utils.load_tokenizer(model_name_or_path, 'left')

In [ ]:
in_file_path = f'{in_dir}/msnap_gpt-5.2_checked_contexts_{model_name}.json'
datas = json_utils.load_json(in_file_path)

print(json_utils.to_str(datas[0]))

In [ ]:
def add_data(datas: list, id, query, prompt, answer_fact, answer_counter):
    data = {
        'id': id,
        'query': query,
        'prompt': prompt,
        'answer_fact': answer_fact,
        'answer_counter': answer_counter
    }

    datas.append(data)

In [ ]:
def make_datas(datas, extract_fact_size, extract_counter_size, extract_max_size=10):
    '''
        fact_size_freq :    {'1': 633, '2': 412, '3': 375, '4': 362, '5': 323, '6': 280, '7': 248, '8': 190, '9': 160, '10': 116}
        counter_size_freq : {'1': 881, '2': 526, '3': 362, '4': 286, '5': 247, '6': 210, '7': 179, '8': 173, '9': 127, '10': 108}
    '''
    datas_f_c, datas_c_f, datas_shuffle = [], [], []

    cnt_ignore, cnt_fact, cnt_counter = 0, 0, 0
    cnt_comb, cnt_comb_all = 0, 0

    for data in datas:
        id = data['id']
        query = data['question']
        answer_fact = data['answer_fact']
        answer_counter = data['answer_counter']
        context_fact_dict: dict = data['contexts_fact']
        context_counter_dict: dict = data['contexts_counter']

        contexts_fact = list(context_fact_dict.values())
        contexts_counter = list(context_counter_dict.values())
        # contexts_fact = [key.replace('context', 'fact') for key in context_fact_dict.keys()]
        # contexts_counter = [key.replace('context', 'counter') for key in context_counter_dict.keys()]

        fact_size = len(contexts_fact)
        counter_size = len(contexts_counter)

        if fact_size < extract_fact_size or counter_size < extract_counter_size:
            cnt_ignore += 1
            continue

        cnt_fact += fact_size
        cnt_counter += counter_size

        combs_contexts_fact = container_utils.get_combinations(contexts_fact, extract_fact_size)
        combs_contexts_counter = container_utils.get_combinations(contexts_counter, extract_counter_size)

        combs_contexts_f_c, combs_contexts_c_f, combs_contexts_shuffle = [], [], []

        for comb_contexts_fact in combs_contexts_fact:
            for comb_contexts_counter in combs_contexts_counter:
                comb_contexts_f_c = comb_contexts_fact + comb_contexts_counter
                comb_contexts_c_f = comb_contexts_counter + comb_contexts_fact
                comb_contexts_shuffle = comb_contexts_counter + comb_contexts_fact
                random.shuffle(comb_contexts_shuffle)

                combs_contexts_f_c.append(comb_contexts_f_c)
                combs_contexts_c_f.append(comb_contexts_c_f)
                combs_contexts_shuffle.append(comb_contexts_shuffle)
        
        cnt_comb_all += len(combs_contexts_shuffle)
        if extract_max_size < len(combs_contexts_shuffle):
            random_indices = random.sample(range(len(combs_contexts_shuffle)), extract_max_size)
            combs_contexts_f_c = [combs_contexts_f_c[i] for i in random_indices]
            combs_contexts_c_f = [combs_contexts_c_f[i] for i in random_indices]
            combs_contexts_shuffle = [combs_contexts_shuffle[i] for i in random_indices]
        
        cnt_comb += len(combs_contexts_shuffle)
        for i in range(len(combs_contexts_shuffle)):
            contexts_f_c = combs_contexts_f_c[i]
            contexts_c_f = combs_contexts_c_f[i]
            contexts_shuffle = combs_contexts_shuffle[i]

            prompt_f_c = msnap_prompts.get_generate_prompt(query, contexts_f_c)
            prompt_c_f = msnap_prompts.get_generate_prompt(query, contexts_c_f)
            prompt_shuffle = msnap_prompts.get_generate_prompt(query, contexts_shuffle)

            add_data(datas_f_c, id, query, prompt_f_c, answer_fact, answer_counter)
            add_data(datas_c_f, id, query, prompt_c_f, answer_fact, answer_counter)
            add_data(datas_shuffle, id, query, prompt_shuffle, answer_fact, answer_counter)

        # print(f'fact_size : {fact_size}, extract_fact_size : {extract_fact_size}')
        # print(f'counter_size : {counter_size}, extract_counter_size : {extract_counter_size}\n')
        # print(f'combs_contexts_fact {len(combs_contexts_fact)} : {combs_contexts_fact}')
        # print(f'combs_contexts_counter {len(combs_contexts_counter)} : {combs_contexts_counter}\n')
        # print(f'combs_contexts_f_c {len(combs_contexts_f_c)} : {combs_contexts_f_c}\n')
        # print(f'combs_contexts_c_f {len(combs_contexts_c_f)} : {combs_contexts_c_f}\n')
        # print(f'combs_contexts_shuffle {len(combs_contexts_shuffle)} : {combs_contexts_shuffle}')
        # break



    print(f'make_datas() data size : {len(datas)}, ext_fact : {extract_fact_size}, ext_counter : {extract_counter_size}, ext_max : {extract_max_size}')
    print(f'make_datas() cnt_ignore : {cnt_ignore}, cnt_fact : {cnt_fact}, cnt_counter : {cnt_counter}')
    print(f'make_datas() cnt_comb : {cnt_comb}/{cnt_comb_all} ({(cnt_comb/cnt_comb_all):.4f})\n')
    print(f'make_datas() datas_f_c size : {len(datas_f_c)}')
    print(f'make_datas() datas_c_f size : {len(datas_c_f)}')
    print(f'make_datas() datas_shuffle size : {len(datas_shuffle)}\n')

    return datas_f_c, datas_c_f, datas_shuffle

In [ ]:
# datas_f_c, datas_c_f, datas_shuffle = make_datas(datas, 2, 5)

# print(f'datas_f_c :\n{datas_f_c[0]}\n\n')
# print(f'datas_c_f :\n{datas_c_f[0]}\n\n')
# print(f'datas_shuffle :\n{datas_shuffle[0]}\n\n')

In [ ]:
def check_targets(datas, batch_size, out_prefix):
    checked_fact, checked_counter, checked_other = [], [], []

    for datas_batch in chunks(datas, batch_size):
        prompts = [data['prompt'] for data in datas_batch]

        generated_texts = model_utils.get_generated_texts(
            model, tokenizer, device,
            prompts, max_seq_length, max_new_tokens
        )

        for data, generated_text in zip(datas_batch, generated_texts):
            answer_fact = data['answer_fact']
            answer_counter = data['answer_counter']
            data['generated_text'] = generated_text

            if model_utils.is_correct(generated_text, answer_fact)[1]:
                checked_fact.append(data)
            elif model_utils.is_correct(generated_text, answer_counter)[1]:
                checked_counter.append(data)
            else:
                checked_other.append(data)



    print(f'check_targets() data size : {len(datas)}\n')

    cnt_fact, cnt_counter, cnt_other = len(checked_fact), len(checked_counter), len(checked_other)
    cnt_all = cnt_fact + cnt_counter + cnt_other

    print(f'check_targets cnt_fact : {cnt_fact}/{cnt_all} ({(cnt_fact/cnt_all):.4f})')
    print(f'check_targets cnt_counter : {cnt_counter}/{cnt_all} ({(cnt_counter/cnt_all):.4f})')
    print(f'check_targets cnt_other : {cnt_other}/{cnt_all} ({(cnt_other/cnt_all):.4f})\n')

    json_utils.write_json(checked_fact, f'{out_prefix}_A-F.json')
    json_utils.write_json(checked_counter, f'{out_prefix}_A-C.json')
    json_utils.write_json(checked_other, f'{out_prefix}_A-O.json')
    print()

In [ ]:
# extract_size_pairs = []

# for extract_fact_size in range(1, 11):
#     for extract_counter_size in range(1, 11):
#         extract_size_pairs.append([extract_fact_size, extract_counter_size])

extract_size_pairs = [[2,8], [8,2], [3,7], [7,3], [1,3], [3,1]]

In [ ]:
for extract_size_pair in extract_size_pairs:
    extract_fact_size, extract_counter_size = extract_size_pair[0], extract_size_pair[1]

    if '3B' in model_name:
        batch_size = 10
    else:
        if (extract_fact_size + extract_counter_size) <= 5:
            batch_size = 5
        elif (extract_fact_size + extract_counter_size) <= 10:
            batch_size = 2
        else:
            batch_size = 1
    
    datas_f_c, datas_c_f, datas_shuffle = make_datas(datas, extract_fact_size, extract_counter_size)

    out_prefix = f'{out_dir}/{model_name}/maked'
    file_prefix = f'msnap_gpt-5.2_maked_targets_{model_name}'

    json_utils.write_json(datas_f_c, f'{out_prefix}/F-C/F-{extract_fact_size}/{file_prefix}_F-C_F-{extract_fact_size}_C-{extract_counter_size}.json')
    json_utils.write_json(datas_c_f, f'{out_prefix}/C-F/F-{extract_fact_size}/{file_prefix}_C-F_F-{extract_fact_size}_C-{extract_counter_size}.json')
    json_utils.write_json(datas_shuffle, f'{out_prefix}/shuffle/F-{extract_fact_size}/{file_prefix}_shuffle_F-{extract_fact_size}_C-{extract_counter_size}.json')
    print()

    out_prefix = f'{out_dir}/{model_name}/checked'
    file_prefix = f'msnap_gpt-5.2_checked_targets_{model_name}'

    check_targets(datas_f_c, batch_size, f'{out_prefix}/F-C/F-{extract_fact_size}/{file_prefix}_F-C_F-{extract_fact_size}_C-{extract_counter_size}')
    check_targets(datas_c_f, batch_size, f'{out_prefix}/C-F/F-{extract_fact_size}/{file_prefix}_C-F_F-{extract_fact_size}_C-{extract_counter_size}')
    check_targets(datas_shuffle, batch_size, f'{out_prefix}/shuffle/F-{extract_fact_size}/{file_prefix}_shuffle_F-{extract_fact_size}_C-{extract_counter_size}')